In [ ]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
import time
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix



In [ ]:
# Cell 2 — Load Data
base = Path.cwd()
if not (base / "data").exists():
    base = base.parent

X_train = pd.read_csv(base / "data/processed/train/X_train.csv")
X_val   = pd.read_csv(base / "data/processed/val/X_val.csv")
X_test  = pd.read_csv(base / "data/processed/test/X_test.csv")

y_train = pd.read_csv(base / "data/processed/train/y_train.csv").values.ravel()
y_val   = pd.read_csv(base / "data/processed/val/y_val.csv").values.ravel()
y_test  = pd.read_csv(base / "data/processed/test/y_test.csv").values.ravel()

feature_names = X_train.columns.tolist()
N_FEATURES = len(feature_names)
print(f"Features: {N_FEATURES}")



In [ ]:
# Cell 3 — Subsample for SA speed
SAMPLE_SIZE = 50000
np.random.seed(42)
idx = np.random.choice(len(X_train), SAMPLE_SIZE, replace=False)
X_sample = X_train.iloc[idx]
y_sample = y_train[idx]
print(f"Using {SAMPLE_SIZE} samples for SA evaluation")



In [ ]:
# Cell 4 — SA Configuration
TEMP_INITIAL = 1.0      # starting temperature
TEMP_MIN     = 0.001    # stop when temperature reaches this
COOLING_RATE = 0.95     # multiply temp by this each iteration (slow cooling)
N_ITERATIONS = 100      # iterations per temperature step
MIN_FEATURES = 5

np.random.seed(42)



In [ ]:
# Cell 5 — SA Helper Functions
def fitness(mask):
    """Evaluate feature subset — return F1 on validation set."""
    selected = np.where(mask == 1)[0]
    if len(selected) < MIN_FEATURES:
        return 0.0

    clf = RandomForestClassifier(
        n_estimators=50,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
    clf.fit(X_sample.iloc[:, selected], y_sample)
    y_pred = clf.predict(X_val.iloc[:, selected])
    return f1_score(y_val, y_pred)

def get_neighbour(mask):
    """Flip 1-3 random bits to create a neighbouring solution."""
    neighbour = mask.copy()
    n_flips = np.random.choice([1, 2, 3])
    flip_idx = np.random.choice(N_FEATURES, n_flips, replace=False)
    for idx in flip_idx:
        neighbour[idx] = 1 - neighbour[idx]

    # ensure minimum features
    if neighbour.sum() < MIN_FEATURES:
        zeros = np.where(neighbour == 0)[0]
        extra = np.random.choice(zeros, MIN_FEATURES - int(neighbour.sum()), replace=False)
        neighbour[extra] = 1
    return neighbour

def acceptance_probability(current_score, new_score, temperature):
    """
    Always accept improvement.
    Accept worse solution with probability e^(delta/T)
    """
    if new_score > current_score:
        return 1.0
    delta = new_score - current_score
    return math.exp(delta / temperature)



In [ ]:
# Cell 6 — Run SA
print("=" * 50)
print("Running Simulated Annealing...")
print("=" * 50)

# Initialise — start with all features
current_mask  = np.ones(N_FEATURES, dtype=int)
current_score = fitness(current_mask)

best_mask  = current_mask.copy()
best_score = current_score

temperature = TEMP_INITIAL
history = []
iteration  = 0
start_time = time.time()

print(f"Initial F1 (all features): {current_score:.4f}")
print(f"Starting temperature: {temperature}")

while temperature > TEMP_MIN:
    for _ in range(N_ITERATIONS):
        neighbour       = get_neighbour(current_mask)
        neighbour_score = fitness(neighbour)

        # Accept or reject
        ap = acceptance_probability(current_score, neighbour_score, temperature)
        if np.random.rand() < ap:
            current_mask  = neighbour
            current_score = neighbour_score

        # Track global best
        if current_score > best_score:
            best_score = current_score
            best_mask  = current_mask.copy()

    iteration += 1
    history.append({
        "iteration": iteration,
        "temperature": temperature,
        "current_f1": current_score,
        "best_f1": best_score,
        "n_features": int(current_mask.sum())
    })

    print(f"Iter {iteration:3d} | Temp: {temperature:.4f} | "
          f"Current F1: {current_score:.4f} | Best F1: {best_score:.4f} | "
          f"Features: {int(current_mask.sum())}")

    temperature *= COOLING_RATE  # cool down

total_time = time.time() - start_time
selected_features = np.where(best_mask == 1)[0]
selected_names    = [feature_names[i] for i in selected_features]

print(f"\nSA completed in {total_time:.1f}s")
print(f"Best F1 (val):     {best_score:.4f}")
print(f"Features selected: {len(selected_features)} / {N_FEATURES}")



In [ ]:
# Cell 7 — Retrain on Full Data with Best Feature Subset
print("\nRetraining on full training set...")
final_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
final_model.fit(X_train.iloc[:, selected_features], y_train)
y_pred = final_model.predict(X_test.iloc[:, selected_features])



In [ ]:
# Cell 8 — Evaluate
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("=" * 45)
print("SA + RF RESULTS")
print("=" * 45)
print(f"Accuracy:            {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision:           {precision_score(y_test, y_pred):.4f}")
print(f"Recall (Det. Rate):  {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:            {f1_score(y_test, y_pred):.4f}")
print(f"False Positive Rate: {fp / (fp + tn):.4f}")
print(f"Features Used:       {len(selected_features)} / {N_FEATURES}")
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nSelected features:\n{selected_names}")



In [ ]:
#Plots
history_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Convergence
axes[0].plot(history_df["iteration"], history_df["best_f1"], "b-", label="Best F1")
axes[0].plot(history_df["iteration"], history_df["current_f1"], "r--", alpha=0.5, label="Current F1")
axes[0].set_xlabel("Iteration")
axes[0].set_ylabel("F1 Score")
axes[0].set_title("SA Convergence Curve")
axes[0].legend()
axes[0].grid(True)

# Temperature decay
axes[1].plot(history_df["iteration"], history_df["temperature"], "orange")
axes[1].set_xlabel("Iteration")
axes[1].set_ylabel("Temperature")
axes[1].set_title("Cooling Schedule")
axes[1].grid(True)

# Feature count over time
axes[2].plot(history_df["iteration"], history_df["n_features"], "g-")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Number of Features")
axes[2].set_title("Feature Count During SA")
axes[2].grid(True)

plt.tight_layout()
plt.savefig("sa_convergence.png", dpi=150)
plt.show()

# Cell 10 — Save Results
sa_results = {
    "method": "SA",
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall": recall_score(y_test, y_pred),
    "f1": f1_score(y_test, y_pred),
    "fpr": fp / (fp + tn),
    "n_features": len(selected_features),
    "runtime_s": total_time
}

pd.DataFrame([sa_results]).to_csv(base / "data/processed/sa_results.csv", index=False)
np.save(base / "data/processed/sa_selected_features.npy", selected_features)
print("SA results saved.")